In [1]:
import torch 
import os
import numpy as np
import torch.nn.functional as F

from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'Data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf100_test_data = datasets.CIFAR100(root = 'Data', train = False, download= True, transform = transforms.ToTensor())

In [4]:
# create validation set
print(len(cf10_training_data))


# print(len(cf100_training_data))

cf10_val_data = random_split(cf10_training_data, [int(len(cf10_training_data)*0.8), int(len(cf10_training_data)*0.2)])
#cf100_val_data = random_split(cf100_training_data, [len(cf100_training_data)*0.8, len(cf100_training_data)*0.2])

50000


In [5]:
print(1)
training_loader = torch.utils.data.DataLoader(cf10_training_data, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_data, batch_size=32, shuffle=True)
testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

1


In [6]:
class LesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = X.view(-1, 16*5*5)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)
        

In [7]:
torch.manual_seed(42)
print(1)
model_1 = LesNet().to(device)

1


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_1.parameters(), lr=0.001)

In [9]:
def training_model(model, nr_epochs):
    losses = []
    for epoch in range(nr_epochs):
        running_loss = 0

        for i , data in enumerate(training_loader):
            inputs, label = data

            if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()

            else:
                model.cpu()

            optimizer.zero_grad()
            outputs = model.forward(inputs)
            loss = criterion(outputs, label)

            

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

           
        print(f"Epoch {epoch+1}, Loss : {running_loss/len(training_loader)}")


            
training_model(model_1, 20)

Epoch 1, Loss : 1.5976553216471705
Epoch 2, Loss : 1.3200814998920194
Epoch 3, Loss : 1.1996841802096718
Epoch 4, Loss : 1.1202609211072965
Epoch 5, Loss : 1.0558771556261175
Epoch 6, Loss : 1.0078685559947294
Epoch 7, Loss : 0.9603208709777508
Epoch 8, Loss : 0.9150990289102665
Epoch 9, Loss : 0.8820213565670826
Epoch 10, Loss : 0.8439992754907846
Epoch 11, Loss : 0.8126174335821423
Epoch 12, Loss : 0.7769998818044852
Epoch 13, Loss : 0.7480353233490856
Epoch 14, Loss : 0.7175656542591917
Epoch 15, Loss : 0.6941623341957118
Epoch 16, Loss : 0.6686419380813245
Epoch 17, Loss : 0.6458462811362949
Epoch 18, Loss : 0.6240061754438256
Epoch 19, Loss : 0.5998347802606059
Epoch 20, Loss : 0.5770629386507542


In [ ]:
# model 2

class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)
        self.conv3 = nn.Conv2d(9, 20, 5, 1 )

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        #add dropuot 
        #self.dropout = nn.Dropout2d(p = 0.3) # 30% chance to be 0

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        #X = F.dropout2d(X, p = 0.3)  #add dropuot 

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        #X = F.dropout2d(X, p = 0.3)  #add dropuot 

        X = F.relu(self.conv3(X))
        X = F.max_pool2d(X, (2,2))

        X = X.view(-1, 16*5*5)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)

model_2 = Model2().to(device)

In [18]:
# testing model 2
training_model(model_2, 20)

Epoch 1, Loss : 2.4612780171026425
Epoch 2, Loss : 2.4613334989212143
Epoch 3, Loss : 2.4612104863748288
Epoch 4, Loss : 2.461199476409248
Epoch 5, Loss : 2.461247670368285
Epoch 6, Loss : 2.461182511470597
Epoch 7, Loss : 2.4612543906299105
Epoch 8, Loss : 2.461218300951801
Epoch 9, Loss : 2.4612320366381684
Epoch 10, Loss : 2.4612592346608753
Epoch 11, Loss : 2.461225244835715
Epoch 12, Loss : 2.461283126399064
Epoch 13, Loss : 2.461326572121677


KeyboardInterrupt: 